# `reducnn` Cats vs. Dogs Research Suite (GitHub Version) 🐈🐕

This notebook executes structural pruning experiments on the **Cats vs. Dogs** dataset using the `reducnn` framework. It is optimized for GitHub/Colab environments.

In [ ]:
# --- STEP 0: GITHUB BOOTLOADER (Colab Optimization) ---
import sys, os

# 1. Clone the repository from GitHub
repo_url = "https://github.com/albertraviss2023/activation-based-pruning.git"
repo_dir = "activation-based-pruning"

if not os.path.exists(repo_dir):
    print(f"🚀 Cloning {repo_url}...")
    !git clone {repo_url}
else:
    print(f"✅ Repository {repo_dir} already exists. Updating...")
    %cd {repo_dir}
    !git pull
    %cd ..

# 2. Environment Setup
os.chdir(repo_dir)
sys.path.insert(0, os.path.abspath("src"))

# 3. Install dependencies and editable package
!pip install -q --upgrade setuptools pip
!pip install -e .

# 4. Load autoreload and Verify

# --- Python 3.12 Compatibility Shim ---
import sys
try:
    import imp
except ImportError:
    from types import ModuleType
    import importlib
    imp = ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp
    print("🛠️ Applied Python 3.12 'imp' shim")

%load_ext autoreload
%autoreload 2
import reducnn
print(f"\n✅ System Ready! Module loaded from: {reducnn.__file__}")

In [ ]:
import torch, torchvision, torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import reducnn as sp
import reducnn.visualization as viz
from reducnn.pruner import ReduCNNPruner
from reducnn.backends.torch_backend import PyTorchAdapter

# Local repository paths
checkpoint_dir = "my_models/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"📁 Local workspace initialized at: {os.getcwd()}")

In [ ]:
# --- Dataset Download and Extraction ---
data_base_dir = "./data/cats_dogs"
os.makedirs(data_base_dir, exist_ok=True)

zip_path = os.path.join(data_base_dir, "kagglecatsanddogs_5340.zip")
if not os.path.exists(os.path.join(data_base_dir, "PetImages")):
    print("📥 Downloading Cats vs Dogs dataset...")
    !wget -O {zip_path} https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
    print("📦 Extracting dataset...")
    !unzip -q {zip_path} -d {data_base_dir}
    print("✅ Dataset ready!")
else:
    print("✅ Dataset already exists.")

In [ ]:
# --- Data Loading with ImageFolder ---
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

pet_images_path = os.path.join(data_base_dir, "PetImages")

# Note: PetImages contains Cat/ and Dog/ folders
# We handle potential corrupt images with a custom loader or just filter
def is_valid_image(path):
    try:
        from PIL import Image
        with Image.open(path) as img:
            img.verify()
        return True
    except:
        return False

full_dataset = torchvision.datasets.ImageFolder(root=pet_images_path, transform=transform, is_valid_file=is_valid_image)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32)

print(f"✅ Loaders ready. Classes: {full_dataset.classes}. Total images: {len(full_dataset)}")

In [ ]:
print('🧪 PYTORCH: ResNet-18 Full Suite (Cats vs Dogs)')
t_res_adapter = PyTorchAdapter(config={'lr': 1e-4, 'input_shape': (3, 128, 128), 'num_classes': 2})
t_res_model = t_res_adapter.get_model('resnet18')

print('1. Establishing ResNet Baseline...')
t_res_adapter.train(t_res_model, train_loader, epochs=2, name='ResNet_Base', val_loader=test_loader)
res_base_acc = t_res_adapter.evaluate(t_res_model, test_loader)
print(f'\n✅ Baseline Accuracy: {res_base_acc:.2f}%')

print('\n2. Performing Structural Surgery (Local 20%)...')
res_surgeon = ReduCNNPruner(method='l1_norm', scope='local')
pruned_res, res_masks, _ = res_surgeon.prune(t_res_model, train_loader, ratio=0.2)

print('\n3. Healing Phase (Fine-tuning)...')
t_res_adapter.train(pruned_res, train_loader, epochs=3, name='ResNet_Heal', val_loader=test_loader)
res_pruned_acc = t_res_adapter.evaluate(pruned_res, test_loader)

print(f'\n✅ ResNet-18 Results (Cats vs Dogs):')
print(f'   Baseline Acc: {res_base_acc:.2f}%')
print(f'   Pruned Acc:   {res_pruned_acc:.2f}%')

viz.plot_layer_sensitivity(res_masks, 'ResNet-18 Pruning Sensitivity (Cats vs Dogs)')